# ApuCast v2.1 en Kaggle con dos GPU T4

Este notebook exporta las tres CNN archivadas del v2, entrena tres ResNet con los datos oficiales y mezcla ambos CSV al 50 %. La configuración completa usa 25 épocas por semilla. Dos procesos independientes entrenan en paralelo, uno por GPU.

En Kaggle, adjunta la competencia **Hackathon IA PUCP 2026** como entrada y selecciona **GPU T4 x2**. Si abres este archivo en VS Code, selecciona el servidor de Kaggle como kernel. El código se obtiene del repositorio del equipo en un commit fijo. Habilita Internet si aún no está clonado en `/kaggle/working`.

## 1. Configuración y verificación

Usa `PRUEBA_RAPIDA = True` para verificar las dos GPU con dos semillas y una época. En modo completo, el archivo de envío queda en `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import torch

PRUEBA_RAPIDA = False
RAIZ = Path("/kaggle/working")
REPO = RAIZ / "ApuCast-Hack-PUCP"
COMMIT = "b4d704e9c281cb8947d8445681814409fdba83d8"
SEMILLAS = (0, 1) if PRUEBA_RAPIDA else (0, 1, 2)
EPOCAS = 1 if PRUEBA_RAPIDA else 25
SALIDAS = RAIZ / ("apucast_prueba" if PRUEBA_RAPIDA else "apucast_v2_1")
TRABAJO = SALIDAS / "resnet"

if torch.cuda.device_count() != 2:
    raise RuntimeError("Selecciona GPU T4 x2 en Kaggle y vuelve a ejecutar la celda.")

for index in range(2):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")

## 2. Código y datos oficiales

El notebook acepta los cuatro archivos desde la entrada de la competencia o desde la carpeta `data` de la copia del equipo en Kaggle. Usa la entrada de la competencia para una ejecución nueva con **Save & Run All**.

In [ ]:
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", "phaletasss-max-v2.1-production",
         "https://github.com/ema28pro/ApuCast-Hack-PUCP.git", str(REPO)],
        check=True,
    )
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if head != COMMIT:
    raise RuntimeError(f"Se esperaba el commit {COMMIT}, pero la copia tiene {head}.")

archivos = ("train.npz", "test.npz", "static.npz", "sample_submission.csv")
entradas = Path("/kaggle/input")
candidatos = [
    p.parent for p in entradas.rglob("train.npz")
    if "hackathon-ia-pucp-2026" in p.parts
]
candidatos.append(REPO / "data")
DATOS = next((p for p in candidatos if all((p / f).is_file() for f in archivos)), None)
if DATOS is None:
    raise FileNotFoundError("Adjunta los cuatro archivos oficiales desde Input > Add Input.")
print(f"Datos: {DATOS}")

TRABAJO.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / "train"))

## 3. Entrena las ResNet en paralelo

La primera T4 entrena las semillas 0 y 2. La segunda entrena la 1. Cada proceso utiliza la arquitectura, el optimizador y el protocolo de la rama v2.1. Guarda un checkpoint por semilla terminada y muestra el avance por época. Si vuelves a ejecutar esta celda en la misma sesión, reutiliza los checkpoints del mismo modo de ejecución.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

worker = r"""
import sys
from pathlib import Path

import numpy as np

from generar_submission_v3 import predict, train_seed
from runtime_apucast import load_starter

repo, data, work = map(Path, sys.argv[1:4])
epochs = int(sys.argv[4])
seeds = [int(value) for value in sys.argv[5:]]
ns = load_starter(repo, data)
hour, tensor = ns["hour"], ns["to_t"]
train_hours = hour(ns["ttr"])
test_hours = hour(ns["tte"])
clim = ns["climatology"](ns["Ytr"], ns["ttr"])
mu = ns["Xtr"].mean(axis=(0, 1, 2))
sd = ns["Xtr"].std(axis=(0, 1, 2))
anomalies = ns["Ytr"] - clim[train_hours]
scale = anomalies.std(axis=(0, 1, 2))
train = tensor((ns["Wtr"] - mu) / sd)
targets = tensor(anomalies / scale)
test = tensor((ns["Wte"] - mu) / sd)
train_hours = tensor(train_hours)
test_reference = tensor(clim[test_hours])
test_hours = tensor(test_hours)
scale_tensor = tensor(scale)

if 0 in seeds:
    np.savez_compressed(work / "preprocesamiento.npz", mu=mu, sd=sd, scale=scale, climatology=clim)
    np.save(work / "horas_test.npy", ns["tte"], allow_pickle=False)

for seed in seeds:
    model = train_seed(ns, seed, epochs, train, train_hours, targets, work / f"resnet_{seed}.pt")
    prediction = predict(model, test, test_hours, test_reference, scale_tensor)
    np.save(work / f"prediccion_{seed}.npy", prediction, allow_pickle=False)
    del model
"""

def entrenar_gpu(dispositivo, semillas):
    visibles = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1").split(",")
    env = {
        **os.environ,
        "CUDA_VISIBLE_DEVICES": visibles[dispositivo],
        "PYTHONPATH": str(REPO / "train"),
        "OMP_NUM_THREADS": "2",
        "MKL_NUM_THREADS": "2",
    }
    comando = [sys.executable, "-u", "-c", worker, str(REPO), str(DATOS), str(TRABAJO),
               str(EPOCAS), *map(str, semillas)]
    print(f"GPU {dispositivo}: semillas {semillas}", flush=True)
    with subprocess.Popen(comando, cwd=REPO, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proceso:
        for linea in proceso.stdout:
            print(f"[GPU {dispositivo}] {linea.rstrip()}", flush=True)
        if proceso.wait():
            raise subprocess.CalledProcessError(proceso.returncode, f"Entrenamiento GPU {dispositivo}")

with ThreadPoolExecutor(max_workers=2) as pool:
    tareas = [pool.submit(entrenar_gpu, gpu, SEMILLAS[gpu::2]) for gpu in range(2)]
    for tarea in tareas:
        tarea.result()

## 4. Exporta y valida los CSV

El modo completo usa los pesos archivados del v2, verifica los manifiestos de ambos componentes y genera `submission.csv`. La prueba rápida genera únicamente `apucast_prueba/resnet.csv` para revisar el flujo sin preparar un envío competitivo.

In [ ]:
from generar_mezcla_v3 import check, read_verified
from generar_submission_v3 import VARIABLES, save_submission

plantilla = pd.read_csv(DATOS / "sample_submission.csv")
horas = np.load(TRABAJO / "horas_test.npy", allow_pickle=False)
predicciones = [np.load(TRABAJO / f"prediccion_{s}.npy", allow_pickle=False) for s in SEMILLAS]
pronostico = np.mean(predicciones, axis=0, dtype=np.float32)
pronostico[..., 0] = pronostico[..., 0].clip(0, 100)

resnet_csv = SALIDAS / "resnet.csv"
resnet_sha = save_submission(plantilla, pronostico, horas, resnet_csv)
resnet_csv.with_suffix(".json").write_text(json.dumps({
    "archivo": resnet_csv.name,
    "sha256": resnet_sha,
    "semillas": list(SEMILLAS),
    "epocas_fijas": EPOCAS,
    "prueba_rapida": PRUEBA_RAPIDA,
    "score_publico": None,
}, indent=2) + "\n", encoding="utf-8")

def ejecutar_script(script, *argumentos):
    subprocess.run([sys.executable, str(REPO / "train" / script), *map(str, argumentos)],
                   cwd=REPO, check=True)

if PRUEBA_RAPIDA:
    print(f"Prueba rápida completada: {resnet_csv}")
else:
    v2_csv = SALIDAS / "v2_top.csv"
    v2_json = v2_csv.with_suffix(".json")
    if not v2_csv.exists() and not v2_json.exists():
        ejecutar_script("exportar_v2.py", "--data", DATOS, "--output", v2_csv)

    final_csv = RAIZ / "submission.csv"
    final_json = final_csv.with_suffix(".json")
    if not final_csv.exists() and not final_json.exists():
        ejecutar_script("generar_mezcla_v3.py", "--data", DATOS, "--v2", v2_csv,
                        "--v2-manifest", v2_json, "--resnet", resnet_csv, "--output", final_csv)

    final, final_sha, registro = read_verified(final_csv, final_json)
    check(final, plantilla)
    _, v2_sha, _ = read_verified(v2_csv, v2_json)
    if [fuente["sha256"] for fuente in registro["fuentes"]] != [v2_sha, resnet_sha]:
        raise RuntimeError("El submission.csv existente corresponde a otra ejecución. Conserva sus archivos en otra carpeta antes de volver a exportar.")
    print(f"Envío listo: {final_csv} | {len(final):,} filas | SHA-256 {final_sha}")

## 5. Entrega

Conserva el CSV, su manifiesto y los pesos ResNet de `apucast_v2_1/resnet`. Para el pitch, presenta el **0,433510** como el score público del archivo v2 archivado. Registra el score de la mezcla cuando Kaggle evalúe el hash de ese envío. Este notebook carga los pesos archivados del v2 y entrena el componente ResNet.